# Plegma Dataset Import Pipeline

Processes raw Plegma per-house CSV files into per-house **Parquet** files ready for model training.

**Pipeline steps:**
1. Load per-house appliance CSVs (monthly files)
2. Merge environmental (weather) data
3. Resample to hourly binary ON/OFF
4. Add time features and save

## Configuration & Imports

In [1]:
import pandas as pd
import numpy as np
import os
import glob

##############
# CHANGE THESE
##############
PLEGMA_ROOT = r"C:\Users\moham\Documents\490 project new\Clean_Dataset"
OUTPUT_DIR  = r"C:\Users\moham\Documents\490 project new\plegma_houses"

os.makedirs(OUTPUT_DIR, exist_ok=True)

APPLIANCE_MAP = {
    'washing_machine': 'elec_clothes_washer_on',
    'fridge':          'elec_refrigerator_on',
    'freezer':         'elec_freezer_on',
    'ac_1':            'elec_cooling_on',
    'ac_2':            'elec_cooling_2_on',
    'boiler':          'elec_hot_water_on',
    'dishwasher':      'elec_dishwasher_on',
    'microwave':       'elec_microwave_on',
    'oven':            'elec_oven_on',
    'tv':              'elec_television_on',
}


## Helper Functions

### `load_metadata` — per-appliance ON/OFF thresholds

In [2]:
def load_metadata(house_path):
    meta_path = os.path.join(house_path, 'Electric_data', 'appliances_metadata.csv')
    defaults  = {
        'washing_machine': 100.0,
        'fridge':           15.0,
        'freezer':          15.0,
        'ac_1':             50.0,
        'ac_2':             50.0,
        'boiler':           50.0,
        'dishwasher':       50.0,
        'microwave':       500.0,
        'oven':            100.0,
        'tv':               15.0,
    }

    if not os.path.exists(meta_path):
        print(f"    [!] No metadata found, using defaults")
        return defaults
    try:
        df         = pd.read_csv(meta_path)
        thresh_col = [c for c in df.columns if 'threshold' in c.lower()]
        name_col   = [c for c in df.columns if 'appliance' in c.lower()]

        if not thresh_col or not name_col:
            return defaults

        result = dict(zip(df[name_col[0]], df[thresh_col[0]]))
        for k, v in defaults.items():
            if k not in result:
                result[k] = v
        return result
    except Exception as e:
        print(f"    [!] Could not read metadata: {e}")
        return defaults


def load_environmental(env_path):
    if not os.path.exists(env_path):
        return None

    try:
        df = pd.read_csv(env_path)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.set_index('timestamp')

        temp_col = None
        hum_col  = None
        for col in df.columns:
            col_lower = col.lower()
            if 'external' in col_lower and 'temp' in col_lower:
                temp_col = col
            if 'external' in col_lower and 'humid' in col_lower:
                hum_col = col

        if temp_col is None:
            return None

        agg = {}
        if temp_col: agg[temp_col] = 'mean'
        if hum_col:  agg[hum_col]  = 'mean'

        df_hourly = df.resample('H')[list(agg.keys())].mean().reset_index()
        rename    = {}
        if temp_col: rename[temp_col] = 'weather_drybulb_temp_c'
        if hum_col:  rename[hum_col]  = 'weather_relative_humidity_pct'
        df_hourly = df_hourly.rename(columns=rename)

        return df_hourly

    except Exception as e:
        print(f"    [!] Could not load environmental data: {e}")
        return None


def process_electric_month(elec_path, thresholds):
    try:
        df = pd.read_csv(elec_path)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.set_index('timestamp')

        drop_cols = ['V', 'A', 'issues', 'P_agg']
        df = df.drop(columns=[c for c in drop_cols if c in df.columns])

        hourly_cols = {}
        for col in df.columns:
            thresh = thresholds.get(col, 15.0)
            hourly_cols[col] = df[col].resample('H').apply(
                lambda x: int((x > thresh).any())
            )

        return pd.DataFrame(hourly_cols).reset_index()

    except Exception as e:
        print(f"    [!] Could not process {elec_path}: {e}")
        return None

### `process_electric_month` — resample to hourly binary

## Main Pipeline

Iterates over every `House_*` folder, processes each monthly CSV, merges weather, adds time features, and writes one Parquet per house.

In [3]:
if __name__ == "__main__":
    house_folders = sorted(glob.glob(os.path.join(PLEGMA_ROOT, 'House_*')))

    if not house_folders:
        raise FileNotFoundError(f"No House_* folders found in {PLEGMA_ROOT}")

    print(f"Found {len(house_folders)} house folders\n")
    summary = []

    for house_path in house_folders:
        house_name = os.path.basename(house_path)
        print(f"{'='*50}")
        print(f"Processing {house_name}")
        print(f"{'='*50}")

        elec_dir = os.path.join(house_path, 'Electric_data')
        env_dir  = os.path.join(house_path, 'Environmental_data')

        if not os.path.exists(elec_dir):
            print(f"  [SKIPPED] No Electric_data folder")
            continue

        thresholds = load_metadata(house_path)
        print(f"  Appliances: {list(thresholds.keys())}")

        elec_files = sorted(glob.glob(os.path.join(elec_dir, '*.csv')))
        elec_files = [f for f in elec_files
                      if 'metadata' not in os.path.basename(f).lower()]

        if not elec_files:
            print(f"  [SKIPPED] No monthly CSV files found")
            continue

        print(f"  Months found: {len(elec_files)}")
        house_months = []

        for elec_path in elec_files:
            month_name = os.path.basename(elec_path).replace('.csv', '')
            print(f"\n  Month: {month_name}")

            df_elec = process_electric_month(elec_path, thresholds)
            if df_elec is None:
                continue

            env_name = month_name.replace('_', '-') + '.csv'
            env_path = os.path.join(env_dir, env_name)
            df_env   = load_environmental(env_path)

            if df_env is not None:
                df_elec['timestamp_h'] = df_elec['timestamp'].dt.floor('H')
                df_env['timestamp_h']  = df_env['timestamp'].dt.floor('H')
                df_elec = df_elec.merge(
                    df_env.drop(columns=['timestamp'], errors='ignore'),
                    on='timestamp_h',
                    how='left'
                ).drop(columns=['timestamp_h'])
                print(f"    Weather merged")
            else:
                print(f"    [!] No environmental data for {month_name}")

            house_months.append(df_elec)

        if not house_months:
            print(f"  [SKIPPED] No valid months")
            continue

        # Combine all months
        df_house = pd.concat(house_months, ignore_index=True)

        # Rename appliances to standard format
        df_house = df_house.rename(columns={
            k: v for k, v in APPLIANCE_MAP.items()
            if k in df_house.columns
        })

        # Identify appliance columns present in this house
        appliance_cols = [v for v in APPLIANCE_MAP.values()
                          if v in df_house.columns]

        # Add time features
        df_house['hour']        = df_house['timestamp'].dt.hour
        df_house['day_of_week'] = df_house['timestamp'].dt.dayofweek
        df_house['is_weekend']  = (df_house['day_of_week'] >= 5).astype(int)
        df_house['month']       = df_house['timestamp'].dt.month

        # Drop columns that are entirely NaN
        before = df_house.shape[1]
        df_house = df_house.dropna(axis=1, how='all')
        after  = df_house.shape[1]
        if before != after:
            print(f"  Dropped {before - after} fully-NaN columns")

        # Drop rows missing weather data
        before_rows = len(df_house)
        df_house = df_house.dropna()
        after_rows = len(df_house)
        dropped = before_rows - after_rows
        if dropped:
            print(f"  Dropped {dropped:,} rows with NaN (missing weather data)")

        if len(df_house) == 0:
            print(f"  [SKIPPED] No rows left after dropping NaN")
            continue

        # Tag
        df_house['house_id'] = house_name
        df_house['source']   = 'plegma'

        # Save per-house file
        out_path = os.path.join(OUTPUT_DIR, f"{house_name}.parquet")
        df_house.to_parquet(out_path, index=False)

        print(f"\n  Saved: {out_path}")
        print(f"  Rows: {len(df_house):,} | Columns: {df_house.shape[1]}")

        for col in appliance_cols:
            on_hrs = df_house[col].sum()
            pct    = on_hrs / len(df_house) * 100
            print(f"    {col}: {on_hrs} hrs ON ({pct:.1f}%)")

        summary.append({
            'house':   house_name,
            'rows':    len(df_house),
            'columns': df_house.shape[1],
            'file':    out_path,
        })

    print(f"\n{'#'*50}")
    print(f"DONE — {len(summary)} houses processed")
    print(f"{'#'*50}")
    for s in summary:
        print(f"  {s['house']}: {s['rows']:,} rows | {s['columns']} cols")

Found 13 house folders

Processing House_01
  Appliances: ['ac_1', 'ac_2', 'boiler', 'fridge', 'washing_machine', 'freezer', 'dishwasher', 'microwave', 'oven', 'tv']
  Months found: 15

  Month: 2022-07


C:\Users\moham\AppData\Local\Temp\ipykernel_20284\616535061.py:87: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly_cols[col] = df[col].resample('H').apply(
C:\Users\moham\AppData\Local\Temp\ipykernel_20284\616535061.py:62: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df.resample('H')[list(agg.keys())].mean().reset_index()
C:\Users\moham\AppData\Local\Temp\ipykernel_20284\1026935217.py:50: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_elec['timestamp_h'] = df_elec['timestamp'].dt.floor('H')
C:\Users\moham\AppData\Local\Temp\ipykernel_20284\1026935217.py:51: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_env['timestamp_h']  = df_env['timestamp'].dt.floor('H')


    Weather merged

  Month: 2022-08


C:\Users\moham\AppData\Local\Temp\ipykernel_20284\616535061.py:87: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly_cols[col] = df[col].resample('H').apply(
C:\Users\moham\AppData\Local\Temp\ipykernel_20284\616535061.py:62: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df.resample('H')[list(agg.keys())].mean().reset_index()
C:\Users\moham\AppData\Local\Temp\ipykernel_20284\1026935217.py:50: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_elec['timestamp_h'] = df_elec['timestamp'].dt.floor('H')
C:\Users\moham\AppData\Local\Temp\ipykernel_20284\1026935217.py:51: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_env['timestamp_h']  = df_env['timestamp'].dt.floor('H')


    Weather merged

  Month: 2022-09


...

C:\Users\moham\AppData\Local\Temp\ipykernel_20284\616535061.py:87: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly_cols[col] = df[col].resample('H').apply(
C:\Users\moham\AppData\Local\Temp\ipykernel_20284\616535061.py:62: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df.resample('H')[list(agg.keys())].mean().reset_index()
C:\Users\moham\AppData\Local\Temp\ipykernel_20284\1026935217.py:50: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_elec['timestamp_h'] = df_elec['timestamp'].dt.floor('H')
C:\Users\moham\AppData\Local\Temp\ipykernel_20284\1026935217.py:51: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_env['timestamp_h']  = df_env['timestamp'].dt.floor('H')


    Weather merged

  Month: 2023-09


C:\Users\moham\AppData\Local\Temp\ipykernel_20284\616535061.py:87: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly_cols[col] = df[col].resample('H').apply(


    Weather merged
  Dropped 8 rows with NaN (missing weather data)

  Saved: C:\Users\moham\Documents\490 project new\plegma_houses\House_13.parquet
  Rows: 7,016 | Columns: 13
    elec_clothes_washer_on: 110 hrs ON (1.6%)
    elec_refrigerator_on: 5495 hrs ON (78.3%)
    elec_cooling_on: 413 hrs ON (5.9%)
    elec_hot_water_on: 1073 hrs ON (15.3%)

##################################################
DONE — 13 houses processed
##################################################
  House_01: 10,608 rows | 14 cols
  House_02: 4,578 rows | 14 cols
  House_03: 10,716 rows | 14 cols
  House_04: 8,521 rows | 14 cols
  House_05: 6,545 rows | 14 cols
  House_06: 4,369 rows | 12 cols
  House_07: 8,008 rows | 14 cols
  House_08: 2,197 rows | 13 cols
  House_09: 5,803 rows | 12 cols
  House_10: 3,900 rows | 15 cols
  House_11: 3,911 rows | 15 cols
  House_12: 4,617 rows | 15 cols
  House_13: 7,016 rows | 13 cols


C:\Users\moham\AppData\Local\Temp\ipykernel_20284\616535061.py:62: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df.resample('H')[list(agg.keys())].mean().reset_index()
C:\Users\moham\AppData\Local\Temp\ipykernel_20284\1026935217.py:50: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_elec['timestamp_h'] = df_elec['timestamp'].dt.floor('H')
C:\Users\moham\AppData\Local\Temp\ipykernel_20284\1026935217.py:51: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_env['timestamp_h']  = df_env['timestamp'].dt.floor('H')
